# N-BEATS Baseline (October 2024)

Core pipeline only: data split -> N-BEATS training -> rolling forecast -> evaluation (MAPE, MAE, RMSE, R2).

No SSA decomposition here, so this notebook calls Darts directly rather than going through `src/` (which only wraps the two hybrid scenarios).

## Setup

In [1]:
import os
print(os.getcwd())
from pathlib import Path
import sys

d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\notebooks


In [2]:
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from pathlib import Path

In [3]:
import os
import sys

print("cwd      :", Path.cwd())
print("resolve  :", Path().resolve())
print("sys.path :", sys.path[:3])  
from pathlib import Path

cwd      : d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\notebooks
resolve  : D:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\notebooks
sys.path : ['D:\\3. TUGAS AKHIR\\5. TA Alfian\\1_REPO_Q1', 'C:\\Users\\LOQ\\AppData\\Roaming\\uv\\python\\cpython-3.10-windows-x86_64-none\\python310.zip', 'C:\\Users\\LOQ\\AppData\\Roaming\\uv\\python\\cpython-3.10-windows-x86_64-none\\DLLs']


In [4]:
print((Path.cwd() / "../data/desember_cleaned_2024.parquet").resolve())
print((Path.cwd() / "../data/desember_cleaned_2024.parquet").exists())

D:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\data\desember_cleaned_2024.parquet
True


In [5]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.5.1+cu121
12.1
True
1
NVIDIA GeForce RTX 2050


## Libs

In [6]:
import warnings
import logging
import json
import numpy as np
import pandas as pd
import torch.nn as nn
from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import CSVLogger
from tqdm import tqdm

from src import evaluate_series, historical_forecast_metrics, print_evaluation_report

warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\.venv\lib\site-packages\lightning_fabric\__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
d:\3. TUGAS AKHIR\5. TA Alfian\1_REPO_Q1\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load data and split (in-sample / out-of-sample)

In [7]:
df = pd.read_parquet("../data/desember_cleaned_2024.parquet")
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])
series = TimeSeries.from_dataframe(df, time_col="DATE_TIME", value_cols=["BEBAN"]).astype(np.float32)

split_date = pd.Timestamp("2024-10-01 00:00:00")
ts_in, ts_out = series.split_before(split_date)
ts_out = ts_out.head(1488)  # 1 month at 30-minute resolution

scaler = Scaler()
ts_in_scaled = scaler.fit_transform(ts_in)
ts_out_scaled = scaler.transform(ts_out)

val_len = 1056
train_series = ts_in_scaled[:-val_len]
val_series = ts_in_scaled[-val_len:]

print(f"In-sample  : {ts_in.start_time()} to {ts_in.end_time()}")
print(f"Out-of-sample (October) : {ts_out.start_time()} to {ts_out.end_time()}")

In-sample  : 2022-01-01 00:00:00 to 2024-09-30 23:30:00
Out-of-sample (October) : 2024-10-01 00:00:00 to 2024-10-31 23:30:00


## 2. Train N-BEATS (Table 8: October configuration)

In [8]:
with open("../configs/nbeats_baseline.json") as f:
    cfg = json.load(f)

logger = CSVLogger("logs_skripsi", name="NBEATS_Tunggal_Final_October")

model = NBEATSModel(
    input_chunk_length=cfg["input_chunk_length"],
    output_chunk_length=cfg["output_chunk_length"],
    generic_architecture=True,
    num_stacks=cfg["num_stacks"],
    num_blocks=cfg["num_blocks"],
    num_layers=cfg["num_layers"],
    layer_widths=cfg["layer_widths"],
    dropout=cfg["dropout"],
    n_epochs=cfg["n_epochs"],
    batch_size=cfg["batch_size"],
    random_state=cfg["random_state"],
    loss_fn=nn.MSELoss(),
    optimizer_kwargs={"lr": cfg["learning_rate"]},
    pl_trainer_kwargs={
        "accelerator": "gpu",
        "logger": logger,
        "callbacks": [EarlyStopping(
            monitor="val_loss",
            patience=cfg["early_stopping_patience"],
            min_delta=cfg["min_delta"],
            mode="min",
        )],
        "enable_progress_bar": True,
    },
)

model.fit(series=train_series, val_series=val_series, verbose=True)
print("Training finished.")

Epoch 40: 100%|██████████| 726/726 [00:32<00:00, 22.10it/s, v_num=2, train_loss=0.000309, val_loss=0.00103] 
Training finished.


## 3. Rolling forecast on the out-of-sample period (Table 2)

No SSA re-decomposition needed each step -- straight `model.predict()` in a walk-forward loop, using Darts directly.

In [ ]:
history = ts_in_scaled
preds_list = []

step_size = 48
total_steps = len(ts_out_scaled)

for i in tqdm(range(0, total_steps, step_size), desc="Rolling forecast (baseline)"):
    pred = model.predict(n=step_size, series=history)
    preds_list.append(pred)

    if i + step_size <= total_steps:
        actual_chunk = ts_out_scaled[i : i + step_size]
        history = history.append(actual_chunk)

final_scaled = concatenate(preds_list)
final_mw = scaler.inverse_transform(final_scaled)

## 4. Evaluation (training / validation / out-of-sample)

In [10]:
ts_in_mw_raw = scaler.inverse_transform(ts_in_scaled)

metrics_train = historical_forecast_metrics(
    model, ts_in_scaled, ts_in_scaled.time_index[336], scaler, ts_in_mw_raw,
)
metrics_val = historical_forecast_metrics(
    model, ts_in_scaled, ts_in_scaled.time_index[-val_len], scaler, ts_in_mw_raw,
)
metrics_test = evaluate_series(ts_out, final_mw)

print_evaluation_report(
    "N-BEATS BASELINE (October 2024)",
    period_labels={
        "training": "Jan 2022 - Aug 2024",
        "validation": "Sep 2024",
        "test": "Oct 2024",
    },
    metrics_by_set={
        "training": metrics_train,
        "validation": metrics_val,
        "test": metrics_test,
    },
)


EVALUATION REPORT: N-BEATS BASELINE (October 2024)
[1] TRAINING (Jan 2022 - Aug 2024)
   - MAPE : 1.52%
   - MAE  : 54.88 MW
   - RMSE : 107.17 MW
   - R2   : 0.95
--------------------------------------------------
[2] VALIDATION (Sep 2024)
   - MAPE : 2.07%
   - MAE  : 88.61 MW
   - RMSE : 131.23 MW
   - R2   : 0.93
--------------------------------------------------
[3] OUT-OF-SAMPLE / TEST (Oct 2024)
   - MAPE : 2.65%
   - MAE  : 120.49 MW
   - RMSE : 206.55 MW
   - R2   : 0.81
